# 07.01_Auco_scanpy_AucoSpatial_Python

空间数据预处理及 TACCO 注释迁移。

- 当前文件：`analysis/07_spatial_analysis/07.01_Auco_scanpy_AucoSpatial_Python.ipynb`
- 原始来源：`Codes/07.01_scanpy_AucoSpatial.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`anndata`, `matplotlib.pyplot`, `numpy`, `os`, `pandas`, `scanpy`, `seaborn`, `tacco`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
import scanpy
print(scanpy.__version__)

### 处理空间组数据

In [ ]:
import scanpy as sc
import numpy as np

In [ ]:
adata_F3 = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/F3_bin50.h5ad")
adata_F3

In [ ]:
adata_D2 = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/D2_bin50.h5ad")
adata_D2

In [ ]:
adata_D2.var

In [ ]:
# 检查是否有重复基因名
duplicates = adata_D2.var['real_gene_name'].duplicated().sum()
print(f"Duplicated gene names: {duplicates}")

In [ ]:
adata_D2.var_names = adata_D2.var['real_gene_name']
adata_D2.var

In [ ]:
adata_D2.obs

In [ ]:
# 二、基础检查与可视化
import scanpy as sc
import matplotlib.pyplot as plt

# 查看前几行元数据
adata_D2.obs.head()

# 查看基因名
adata_D2.var.head()

# 空间坐标检查
adata_D2.obsm['spatial'][:5]

# 绘制空间分布图
sc.pl.embedding(adata_D2, basis='spatial', color=None, title='Spatial Bins', s=5)


In [ ]:
# 查看是否已经标准化
X = adata_D2.X.toarray()
print("Value type:", X.dtype)
print("Min:", np.min(X))
print("Max:", np.max(X))
print("Mean:", np.mean(X))
print("Median:", np.median(X))
print("Unique nonzero values:", np.unique(X[X > 0])[:10])

In [ ]:
adata_D2

In [ ]:
# 保留原始 count 数据
adata_D2.layers["counts"] = adata_D2.X.copy()
adata_D2

In [ ]:
# 三、预处理流程（适用于 Stereo-seq）
sc.pp.filter_cells(adata_D2, min_genes=10)
sc.pp.filter_genes(adata_D2, min_cells=10)
# 2.标准化与对数转换
sc.pp.normalize_total(adata_D2, target_sum=1e4)
sc.pp.log1p(adata_D2)
# 3.高变基因筛选与PCA降维
sc.pp.highly_variable_genes(adata_D2, n_top_genes=1000, flavor='seurat_v3')
adata_D2

In [ ]:
# 保存原始数据
adata_D2.raw = adata_D2.copy()
adata_D2.X.toarray()

In [ ]:
# 查看是否已经标准化
X = adata_D2.X.toarray()
print("Value type:", X.dtype)
print("Min:", np.min(X))
print("Max:", np.max(X))
print("Mean:", np.mean(X))
print("Median:", np.median(X))
print("Unique nonzero values:", np.unique(X[X > 0])[:10])

In [ ]:
sc.pl.highly_variable_genes(adata_D2)

In [ ]:
sc.tl.pca(adata_D2, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata_D2, n_pcs=25, log=True)

In [ ]:
# adata_D2 = adata_D2[:, adata_D2.var.highly_variable]
# sc.pp.scale(adata_F3, max_value=10)
# sc.pp.scale(adata_D2, zero_center=True, max_value=10)
# sc.tl.pca(adata_D2, svd_solver='arpack')
# 4.邻近图与聚类
sc.pp.neighbors(adata_D2, n_neighbors=8, n_pcs=10)

In [ ]:
sc.tl.umap(adata_D2)

In [ ]:
sc.tl.leiden(adata_D2, resolution=0.3)

In [ ]:
# 可视化聚类结果
# UMAP
sc.pl.umap(adata_D2, color=['leiden'])
# 空间聚类
sc.pl.embedding(adata_D2, basis='spatial', color='leiden', s=8)

In [ ]:
adata_D2

In [ ]:
adata_D2 = adata_D2.raw.to_adata()
adata_D2

In [ ]:
sc.pl.umap(adata_D2, color=["XLOC-019965#SYT14-HUMAN#Q8NB59"])

In [ ]:
sc.pl.violin(
    adata_D2,
    keys="XLOC-019965#SYT14-HUMAN#Q8NB59",
    groupby="leiden",      # 按聚类分组（或换成其他列名）
    jitter=0.4,            # 每个点的抖动幅度
    rotation=45,           # x轴标签旋转角度
    stripplot=True,        # 在小提琴上叠加散点
)

In [ ]:
genes = [
    "XLOC-019965#SYT14-HUMAN#Q8NB59"
]
sc.pl.embedding(
    adata_D2,
    basis='spatial',
    color=genes,
    s=8,
    cmap='viridis'
)


### 细胞类型迁移Overlap of DEGs

In [ ]:
import scanpy as sc

adata_sc = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellAnnotation/Auco.normalized.annotated.h5ad")
sc.tl.rank_genes_groups(
    adata_sc,
    # groupby="Sub_cell_type",
    groupby="Broad_cell_type",
    method="wilcoxon",
    n_genes=5000
)
sc_genes = adata_sc.uns["rank_genes_groups"]


In [ ]:
# adata_st = sc.read_h5ad("D2_bin50.h5ad")
adata_st = adata_D2.copy()
sc.tl.rank_genes_groups(
    adata_st,
    groupby="leiden",
    method="wilcoxon",
    n_genes=5000
)
st_genes = adata_st.uns["rank_genes_groups"]


In [ ]:
import pandas as pd

def get_deg_dict(result, top_n=200):
    deg_dict = {}
    for grp in result["names"].dtype.names:
        deg_dict[grp] = pd.DataFrame({
            "gene": result["names"][grp],
            "pval": result["pvals_adj"][grp],
            "logfc": result["logfoldchanges"][grp]
        }).query("pval < 0.05").head(top_n)["gene"].tolist()
    return deg_dict

sc_deg = get_deg_dict(sc_genes, top_n=2000)
st_deg = get_deg_dict(st_genes, top_n=2000)


In [ ]:
import numpy as np
celltypes = list(sc_deg.keys())
clusters = list(st_deg.keys())

jaccard = np.zeros((len(celltypes), len(clusters)))
for i, ct in enumerate(celltypes):
    for j, cl in enumerate(clusters):
        inter = len(set(sc_deg[ct]) & set(st_deg[cl]))
        union = len(set(sc_deg[ct]) | set(st_deg[cl]))
        jaccard[i, j] = inter / union

import pandas as pd
jaccard_df = pd.DataFrame(jaccard, index=celltypes, columns=clusters)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.heatmap(jaccard_df, cmap="YlGnBu", annot=True, fmt=".2f")
plt.title("Jaccard Similarity between scRNA-seq Cell Types and Spatial Clusters")
plt.xlabel("Spatial clusters")
plt.ylabel("Cell types")
plt.show()


### 细胞类型迁移TACCO

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import tacco as tc

In [ ]:
# 单细胞
adata_Auco_annotated = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellAnnotation/Auco.normalized.annotated.h5ad")
adata_Auco_annotated

In [ ]:
adata_Auco_annotated.X = adata_Auco_annotated.layers["counts"].copy()

In [ ]:
adata_Auco_annotated.X.toarray()

In [ ]:
# 空间组
adata_D2 = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/D2_bin50.h5ad")
adata_D2

In [ ]:
adata_D2.X.toarray()

In [ ]:
adata_D2.var_names = adata_D2.var['real_gene_name']
adata_D2.var

In [ ]:
adata_D2.var.index.to_series().to_csv(
    "/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/D2_bin50.genes.txt",
    index=False,
    header=False
)

In [ ]:
sc.pp.filter_cells(adata_D2, min_genes=80)
sc.pp.filter_genes(adata_D2, min_cells=10)

In [ ]:
# ------------------------------
# 1. 加载单细胞与空间组数据
# ------------------------------
adata_sc = adata_Auco_annotated.copy()
adata_sp = adata_D2.copy()

print("Single-cell data:", adata_sc)
print("Spatial data:", adata_sp)

In [ ]:
# ------------------------------
# 2. 准备输入
# ------------------------------
# 假设单细胞的细胞类型标签在 adata_sc.obs['celltype'] 或 ['CellType'] 中
celltype_key = 'Broad_cell_type'

# 检查表达矩阵一致性
shared_genes = list(set(adata_sc.var_names) & set(adata_sp.var_names))
print(f"共有基因 {len(shared_genes)} 个可用于匹配")

adata_sc = adata_sc[:, shared_genes].copy()
adata_sp = adata_sp[:, shared_genes].copy()

In [ ]:
# ------------------------------
# 3. 计算单细胞中各细胞类型的先验概率
# ------------------------------
prob = adata_sc.obs[celltype_key].value_counts() / adata_sc.n_obs
print("细胞类型先验分布：")
print(prob)

In [ ]:
# ------------------------------
# 4. 准备空间组结构
# ------------------------------
if 'X_spatial' not in adata_sp.obsm:
    adata_sp.obsm['X_spatial'] = adata_sp.obsm['spatial']

adata_sp.layers['data'] = adata_sp.X.copy()
adata_sp.X = adata_sp.layers['data']

In [ ]:
# ------------------------------
# 5. 运行 TACCO 映射注释
# ------------------------------
outpath = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_D2"
os.makedirs(outpath, exist_ok=True)

# 确保矩阵为浮点型
if not np.issubdtype(adata_sc.X.dtype, np.floating):
    adata_sc.X = adata_sc.X.astype(float)
if not np.issubdtype(adata_sp.X.dtype, np.floating):
    adata_sp.X = adata_sp.X.astype(float)


adata_sp = tc.tl.annotate(
    adata_sp,
    adata_sc,
    annotation_key=celltype_key,
    result_key=f'pred_{celltype_key}',
    annotation_prior=prob,
    verbose=True,
    # assume_valid_counts=True  # ✅ 允许非整数输入
)


In [ ]:
# # ------------------------------
# # 6. 保存预测结果
# # ------------------------------
# pred = adata_sp.obsm[f'pred_{celltype_key}'].copy()
# pred[f'pred_{celltype_key}'] = pred.idxmax(1)
# pred[f'pred_{celltype_key}_score'] = pred.max(1)

# adata_sp.obs[f'pred_{celltype_key}'] = pred[f'pred_{celltype_key}']
# adata_sp.obs[f'pred_{celltype_key}_score'] = pred[f'pred_{celltype_key}_score']

# pred.to_csv(os.path.join(outpath, f"D2_pred_{celltype_key}.csv.gz"))
# adata_sp.write(os.path.join(outpath, "D2_mapped.h5ad"))


# ------------------------------
# 6. 保存预测结果（修正版）
# ------------------------------
pred = adata_sp.obsm[f'pred_{celltype_key}'].copy()

# ✅ 只保留数值列（忽略 Broad_cell_type / pred_Broad_cell_type）
numeric_cols = pred.select_dtypes(include=[np.number]).columns
pred_numeric = pred[numeric_cols]

# ✅ 计算最大概率对应的细胞类型与分数
pred[f'pred_{celltype_key}'] = pred_numeric.idxmax(1)
pred[f'pred_{celltype_key}_score'] = pred_numeric.max(1)

# 写回到 obs
adata_sp.obs[f'pred_{celltype_key}'] = pred[f'pred_{celltype_key}']
adata_sp.obs[f'pred_{celltype_key}_score'] = pred[f'pred_{celltype_key}_score']

# 保存结果
outpath = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_D2"
os.makedirs(outpath, exist_ok=True)
pred.to_csv(os.path.join(outpath, f"D2_pred_{celltype_key}.csv"))
adata_sp.write(os.path.join(outpath, "D2_mapped.h5ad"))

In [ ]:
print(pred_numeric.columns)


In [ ]:
pred[['pred_Broad_cell_type', 'pred_Broad_cell_type_score']].head()


In [ ]:
# ------------------------------
# 7. 绘图展示
# ------------------------------
sc.settings.figdir = outpath

# 总体预测分布
sc.pl.embedding(
    adata_sp,
    basis='X_spatial',
    color=f'pred_{celltype_key}',
    title='Spatial Bins',
    s=5,
    save="_predicted_types.png"
)
# sc.pl.spatial(
#     adata_sp,
#     color=f'pred_{celltype_key}',
#     spot_size=1,
#     title='Predicted cell types',
#     save="_predicted_types.png"
# )

In [ ]:
# ------------------------------
# 8. 分面绘图函数
# ------------------------------
def cluster_small_multiples(adata, clust_key, ncol=6, nrow=None, size=10, frameon=False, basis='spatial', **kwargs):
    tmp = adata.copy()
    categories = adata.obs[clust_key].astype('category').cat.categories
    if nrow is None:
        nrow = int(np.ceil(len(categories) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow))
    axs = axs.flatten()

    for i, clust in enumerate(categories):
        ax = axs[i]
        tmp.obs['__highlight__'] = (adata.obs[clust_key] == clust).astype('category')
        # sc.pl.spatial(
        #     tmp,
        #     color='__highlight__',
        #     ax=ax,
        #     show=False,
        #     title=clust,
        #     size=size,
        #     frameon=frameon,
        #     **kwargs)
        sc.pl.embedding(
            tmp,
            basis='X_spatial',
            color='__highlight__',
            ax=ax,
            show=False,
            title=clust,
            size=size,
            frameon=frameon,
            **kwargs)
    for j in range(i + 1, len(axs)):
        fig.delaxes(axs[j])
    plt.tight_layout()
    return fig

fig = cluster_small_multiples(adata_sp, clust_key=f'pred_{celltype_key}', basis="spatial")
fig.savefig(f"{outpath}/D2_pred_{celltype_key}_split.png", dpi=300)

In [ ]:
adata_sp

In [ ]:
adata_sp.X.toarray()

In [ ]:
# 查看是否已经标准化
X = adata_sp.X.toarray()
print("Value type:", X.dtype)
print("Min:", np.min(X))
print("Max:", np.max(X))
print("Mean:", np.mean(X))
print("Median:", np.median(X))
print("Unique nonzero values:", np.unique(X[X > 0])[:10])

In [ ]:
sc.pp.normalize_total(adata_sp, target_sum=1e4)
sc.pp.log1p(adata_sp)

In [ ]:
sc.pp.highly_variable_genes(adata_sp, n_top_genes=1000, flavor='seurat_v3')

In [ ]:
adata_sp

In [ ]:
sc.pl.highly_variable_genes(adata_sp)

In [ ]:
sc.tl.pca(adata_sp, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata_sp, n_pcs=50, log=True)

In [ ]:
sc.pp.neighbors(adata_sp, n_neighbors=8, n_pcs=10)

In [ ]:
sc.tl.umap(adata_sp)

In [ ]:
sc.tl.leiden(adata_sp, resolution=0.3)

In [ ]:
# 可视化聚类结果
# UMAP
sc.pl.umap(adata_sp, color=['leiden'])
# 空间聚类
sc.pl.embedding(adata_sp, basis='spatial', color='leiden', s=8)

In [ ]:
adata_sp

In [ ]:
sc.pl.umap(adata_sp, color=['leiden', 'pred_Broad_cell_type'])

In [ ]:
adata_sp.obs['pred_Broad_cell_type'].value_counts()

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata_sp, groupby="pred_Broad_cell_type", method="wilcoxon")

In [ ]:
# 绘制每组前 3 个 marker 基因的 dotplot
sc.pl.rank_genes_groups_dotplot(
    adata_sp,
    groupby='pred_Broad_cell_type',
    n_genes=5,
    standard_scale='var',
    show=True,
    swap_axes=True,
    color_map='RdBu_r',
    figsize=(14, 12)
)

In [ ]:
adata_sp.var

In [ ]:
adata_sp.var_names

In [ ]:
adata_sp.raw.var_names

In [ ]:
sc.pl.violin(
    adata_sp,
    keys="XLOC-013506",
    groupby="pred_Broad_cell_type",      # 按聚类分组（或换成其他列名）
    jitter=0.4,            # 每个点的抖动幅度
    rotation=45,           # x轴标签旋转角度
    stripplot=True,        # 在小提琴上叠加散点
)

In [ ]:
sc.pl.violin(
    adata_sp,
    keys="XLOC-025117",
    groupby="pred_Broad_cell_type",      # 按聚类分组（或换成其他列名）
    jitter=0.4,            # 每个点的抖动幅度
    rotation=45,           # x轴标签旋转角度
    stripplot=True,        # 在小提琴上叠加散点
)

In [ ]:

genes = [
    "XLOC-013506", "XLOC-025117"
]

sc.pl.embedding(
    adata_sp,
    basis='spatial',
    color=genes,
    s=8,
    cmap='magma'
)


### TACCO 处理 A04233F3

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import tacco as tc

In [ ]:
# 单细胞
adata_Auco_annotated = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellAnnotation/Auco.normalized.annotated.h5ad")
adata_Auco_annotated

In [ ]:
adata_Auco_annotated.X = adata_Auco_annotated.layers["counts"].copy()

In [ ]:
adata_Auco_annotated.X.toarray()

In [ ]:
# 空间组
adata_F3 = sc.read_h5ad("/share/home/zhangze/zz/NeuralOrigin/Data/04.SpatialTranscriptomicsProcessing/F3_bin50.h5ad")
adata_F3

In [ ]:
adata_F3.X.toarray()

In [ ]:
adata_F3.var_names = adata_F3.var['real_gene_name']
adata_F3.var

In [ ]:
sc.pp.filter_cells(adata_F3, min_genes=80)
sc.pp.filter_genes(adata_F3, min_cells=10)

In [ ]:
# ------------------------------
# 1. 加载单细胞与空间组数据
# ------------------------------
adata_sc = adata_Auco_annotated.copy()
adata_sp = adata_F3.copy()

print("Single-cell data:", adata_sc)
print("Spatial data:", adata_sp)

In [ ]:
# ------------------------------
# 2. 准备输入
# ------------------------------
# 假设单细胞的细胞类型标签在 adata_sc.obs['celltype'] 或 ['CellType'] 中
celltype_key = 'Broad_cell_type'

# 检查表达矩阵一致性
shared_genes = list(set(adata_sc.var_names) & set(adata_sp.var_names))
print(f"共有基因 {len(shared_genes)} 个可用于匹配")

adata_sc = adata_sc[:, shared_genes].copy()
adata_sp = adata_sp[:, shared_genes].copy()

In [ ]:
# ------------------------------
# 3. 计算单细胞中各细胞类型的先验概率
# ------------------------------
prob = adata_sc.obs[celltype_key].value_counts() / adata_sc.n_obs
print("细胞类型先验分布：")
print(prob)

In [ ]:
# ------------------------------
# 4. 准备空间组结构
# ------------------------------
if 'X_spatial' not in adata_sp.obsm:
    adata_sp.obsm['X_spatial'] = adata_sp.obsm['spatial']

adata_sp.layers['data'] = adata_sp.X.copy()
adata_sp.X = adata_sp.layers['data']

In [ ]:
# ------------------------------
# 5. 运行 TACCO 映射注释
# ------------------------------
outpath = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_F3"
os.makedirs(outpath, exist_ok=True)

# 确保矩阵为浮点型
if not np.issubdtype(adata_sc.X.dtype, np.floating):
    adata_sc.X = adata_sc.X.astype(float)
if not np.issubdtype(adata_sp.X.dtype, np.floating):
    adata_sp.X = adata_sp.X.astype(float)


adata_sp = tc.tl.annotate(
    adata_sp,
    adata_sc,
    annotation_key=celltype_key,
    result_key=f'pred_{celltype_key}',
    annotation_prior=prob,
    verbose=True,
    # assume_valid_counts=True  # ✅ 允许非整数输入
)


In [ ]:
# # ------------------------------
# # 6. 保存预测结果
# # ------------------------------
# pred = adata_sp.obsm[f'pred_{celltype_key}'].copy()
# pred[f'pred_{celltype_key}'] = pred.idxmax(1)
# pred[f'pred_{celltype_key}_score'] = pred.max(1)

# adata_sp.obs[f'pred_{celltype_key}'] = pred[f'pred_{celltype_key}']
# adata_sp.obs[f'pred_{celltype_key}_score'] = pred[f'pred_{celltype_key}_score']

# pred.to_csv(os.path.join(outpath, f"D2_pred_{celltype_key}.csv.gz"))
# adata_sp.write(os.path.join(outpath, "D2_mapped.h5ad"))


# ------------------------------
# 6. 保存预测结果（修正版）
# ------------------------------
pred = adata_sp.obsm[f'pred_{celltype_key}'].copy()

# ✅ 只保留数值列（忽略 Broad_cell_type / pred_Broad_cell_type）
numeric_cols = pred.select_dtypes(include=[np.number]).columns
pred_numeric = pred[numeric_cols]

# ✅ 计算最大概率对应的细胞类型与分数
pred[f'pred_{celltype_key}'] = pred_numeric.idxmax(1)
pred[f'pred_{celltype_key}_score'] = pred_numeric.max(1)

# 写回到 obs
adata_sp.obs[f'pred_{celltype_key}'] = pred[f'pred_{celltype_key}']
adata_sp.obs[f'pred_{celltype_key}_score'] = pred[f'pred_{celltype_key}_score']

# 保存结果
outpath = "/share/home/zhangze/zz/NeuralOrigin/Data/07.SpatialTranscriptomicsAnalysis/ST_annotated/Tacco_Auco_F3"
os.makedirs(outpath, exist_ok=True)
pred.to_csv(os.path.join(outpath, f"F3_pred_{celltype_key}.csv"))
adata_sp.write(os.path.join(outpath, "F3_mapped.h5ad"))

In [ ]:
print(pred_numeric.columns)


In [ ]:
pred[['pred_Broad_cell_type', 'pred_Broad_cell_type_score']].head()

In [ ]:
# ------------------------------
# 7. 绘图展示
# ------------------------------
sc.settings.figdir = outpath

# 总体预测分布
sc.pl.embedding(
    adata_sp,
    basis='X_spatial',
    color=f'pred_{celltype_key}',
    title='Spatial Bins',
    s=5,
    save="_predicted_types.png"
)
# sc.pl.spatial(
#     adata_sp,
#     color=f'pred_{celltype_key}',
#     spot_size=1,
#     title='Predicted cell types',
#     save="_predicted_types.png"
# )

In [ ]:
# ------------------------------
# 8. 分面绘图函数
# ------------------------------
def cluster_small_multiples(adata, clust_key, ncol=6, nrow=None, size=10, frameon=False, basis='spatial', **kwargs):
    tmp = adata.copy()
    categories = adata.obs[clust_key].astype('category').cat.categories
    if nrow is None:
        nrow = int(np.ceil(len(categories) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow))
    axs = axs.flatten()

    for i, clust in enumerate(categories):
        ax = axs[i]
        tmp.obs['__highlight__'] = (adata.obs[clust_key] == clust).astype('category')
        # sc.pl.spatial(
        #     tmp,
        #     color='__highlight__',
        #     ax=ax,
        #     show=False,
        #     title=clust,
        #     size=size,
        #     frameon=frameon,
        #     **kwargs)
        sc.pl.embedding(
            tmp,
            basis='X_spatial',
            color='__highlight__',
            ax=ax,
            show=False,
            title=clust,
            size=size,
            frameon=frameon,
            **kwargs)
    for j in range(i + 1, len(axs)):
        fig.delaxes(axs[j])
    plt.tight_layout()
    return fig

fig = cluster_small_multiples(adata_sp, clust_key=f'pred_{celltype_key}', basis="spatial")
fig.savefig(f"{outpath}/F3_pred_{celltype_key}_split.png", dpi=300)

In [ ]:
adata_sp

In [ ]:
adata_sp.X.toarray()

In [ ]:
# 查看是否已经标准化
X = adata_sp.X.toarray()
print("Value type:", X.dtype)
print("Min:", np.min(X))
print("Max:", np.max(X))
print("Mean:", np.mean(X))
print("Median:", np.median(X))
print("Unique nonzero values:", np.unique(X[X > 0])[:10])

In [ ]:
sc.pp.normalize_total(adata_sp, target_sum=1e4)
sc.pp.log1p(adata_sp)

In [ ]:
sc.pp.highly_variable_genes(adata_sp, n_top_genes=1000, flavor='seurat_v3')

In [ ]:
adata_sp

In [ ]:
sc.pl.highly_variable_genes(adata_sp)

In [ ]:
sc.tl.pca(adata_sp, svd_solver='arpack')

In [ ]:
sc.pl.pca_variance_ratio(adata_sp, n_pcs=50, log=True)

In [ ]:
sc.pp.neighbors(adata_sp, n_neighbors=8, n_pcs=10)

In [ ]:
sc.tl.umap(adata_sp)

In [ ]:
sc.tl.leiden(adata_sp, resolution=0.3)

In [ ]:
# 可视化聚类结果
# UMAP
sc.pl.umap(adata_sp, color=['leiden'])
# 空间聚类
sc.pl.embedding(adata_sp, basis='spatial', color='leiden', s=8)

In [ ]:
adata_sp

In [ ]:
sc.pl.umap(adata_sp, color=['leiden', 'pred_Broad_cell_type'])

In [ ]:
adata_sp.obs['pred_Broad_cell_type'].value_counts()

In [ ]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata_sp, groupby="pred_Broad_cell_type", method="wilcoxon")

In [ ]:
# 绘制每组前 3 个 marker 基因的 dotplot
sc.pl.rank_genes_groups_dotplot(
    adata_sp,
    groupby='pred_Broad_cell_type',
    n_genes=5,
    standard_scale='var',
    show=True,
    swap_axes=True,
    color_map='RdBu_r',
    figsize=(14, 12)
)